# Notebook 03 - Indice Macroeconomico Argentina (IMA)

Indice mensual que resume **que tan bien o mal esta la macroeconomia** argentina desde 2004.

**Lectura:** 0 = situacion tipica del periodo 2004-hoy; +1 = un desvio "normal" mejor que lo tipico;
-1 = un desvio peor. Cada variable se lleva a un z-score robusto (mediana y rango intercuartil), con signo tal que
**mas alto = mejor**, recortado a +/-3.

### Pilares y variables (16)
| Pilar | Variables | Origen |
|---|---|---|
| Actividad | EMAE desest. (var. i.a.); recaudacion real IVA + Deb./Cred. (trim. movil, var. i.a.) | INDEC; **repo (IMIG)** |
| Empleo e ingresos | Asalariados privados registrados SIPA (var. i.a.); salario real registrado (var. i.a.); desocupacion | datos.gob.ar |
| Precios | Inflacion 3 meses anualizada; inflacion esperada 12 m (REM) | **repo (IPC)** + BCRA |
| Fiscal | Resultado primario 12 m / PIB; intereses / ingresos totales 12 m | **repo (IMIG)** + PIB INDEC |
| Externo | Reservas brutas en meses de importaciones; saldo comercial 12 m; brecha CCL/oficial; desalineamiento del TCRM | BCRA, INDEC, argentinadatos |
| Financiero | Riesgo pais; credito real al sector privado (var. i.a.); desvio de la tasa real de su nivel neutral | argentinadatos, BCRA |

**Indice** = promedio simple de los pilares disponibles (cada pilar = promedio de sus variables).
El pilar Fiscal arranca en dic-2019 (IMIG del repo desde 2019); por eso tambien se reporta el
**indice sin fiscal**, comparable en toda la serie.

**Datos:** `data/reference/macro_mensual.csv` (se actualiza con `python scripts/actualizar_macro.py`),
`data/reference/IPC.xlsx` y `output/imig_consolidado.csv`.

### Descarga (al ejecutar todo)
`indice_macro.zip` = 4 graficos PNG + `indice_macro.xlsx`
(Leeme | Indice | Z_variables | Variables | Metodologia | Por_gobierno | Datos_crudos)

In [ ]:
# ── Celda 1: Dependencias y parametros ───────────────────────────────
import sys, io, zipfile, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

DPI         = 600
GRAFICOS    = []
INICIO      = pd.Timestamp('2004-01-01')  # comienzo del indice (EMAE, REM, ITCRM disponibles)
CLIP_Z      = 3.0    # tope de cada z-score: un outlier (2020, 2023) no domina el indice
FFILL_MAX   = 3      # meses que se arrastra el ultimo dato de una serie que se publica con rezago
MIN_PILARES = 4      # pilares minimos con dato para calcular el indice
MIN_OBS_FIN = 0.6    # ultimo mes del indice: al menos 60% de las variables con dato propio (no arrastrado)
TASA_REAL_NEUTRA = 2.0   # % anual; se penaliza la distancia de la tasa real a este nivel
INICIO_CEPO = pd.Timestamp('2011-11-01')  # antes no habia control de cambios -> brecha = 0

GOBIERNOS = [  # (inicio, etiqueta)
    ('2003-05-25', 'N. Kirchner'), ('2007-12-10', 'C. Fernandez'), ('2011-12-10', 'C. Fernandez II'),
    ('2015-12-10', 'Macri'), ('2019-12-10', 'A. Fernandez'), ('2023-12-10', 'Milei')]

PILARES = ['Actividad', 'Empleo e ingresos', 'Precios', 'Fiscal', 'Externo', 'Financiero']
COLOR_PILAR = {'Actividad': '#1a5276', 'Empleo e ingresos': '#16a085', 'Precios': '#c0392b',
               'Fiscal': '#8e44ad', 'Externo': '#d68910', 'Financiero': '#5d6d7e'}

plt.rcParams['figure.dpi']        = 100
plt.rcParams['savefig.dpi']       = DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size']         = 9

def save_fig(fig, nombre):
    fname = f'{nombre}.png'
    fig.savefig(fname, dpi=DPI, bbox_inches='tight')
    GRAFICOS.append(fname)
    print(f'  Guardado: {fname}')

def sombrear_gobiernos(ax, desde, hasta, etiquetas=True):
    lims = [pd.Timestamp(g[0]) for g in GOBIERNOS] + [hasta]
    for i, (ini, nombre) in enumerate(GOBIERNOS):
        a, b = max(pd.Timestamp(ini), desde), min(lims[i + 1], hasta)
        if a >= b:
            continue
        if i % 2 == 0:
            ax.axvspan(a, b, color='#7f8c8d', alpha=0.08, lw=0)
        if etiquetas:
            ax.text(a + (b - a) / 2, 1.01, nombre, transform=ax.get_xaxis_transform(),
                    ha='center', va='bottom', fontsize=8, color='#555')

In [ ]:
# ── Celda 2: Carga de datos ──────────────────────────────────────────
REPO     = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'
REPO_BIN = 'https://github.com/santiagoriverti/cuentas_publicas/raw/main'

macro = pd.read_csv(f'{REPO}/data/reference/macro_mensual.csv', parse_dates=['fecha'], index_col='fecha')
df_imig = (pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])
             .drop_duplicates(subset=['fecha', 'concepto_codigo', 'nivel_jerarquia']))

ipc_raw = pd.read_excel(io.BytesIO(requests.get(f'{REPO_BIN}/data/reference/IPC.xlsx').content),
                        usecols=['date', 'Nivel general'])
ipc_indec = ipc_raw.set_index(pd.to_datetime(ipc_raw['date']).dt.to_period('M').dt.to_timestamp())['Nivel general'].sort_index()

IDX = pd.date_range('2003-01-01', max(macro.index.max(), ipc_indec.index.max()), freq='MS')
macro = macro.reindex(IDX)

# IPC empalmado: INDEC (dic-2016=100) desde 2017; hacia atras se encadena la inflacion mensual
# que publica el BCRA (2007-2015 = periodo de intervencion del INDEC: tomar con cautela).
infl_bcra = macro['inflacion_mensual'].dropna()
nivel, previos = ipc_indec.iloc[0], {}
for d in sorted(infl_bcra.index[infl_bcra.index <= ipc_indec.index[0]], reverse=True):
    nivel = nivel / (1 + infl_bcra[d] / 100)
    previos[d - pd.DateOffset(months=1)] = nivel
ipc = pd.concat([pd.Series(previos), ipc_indec]).sort_index().reindex(IDX)

def imig_serie(cod, nivel):
    m = (df_imig['concepto_codigo'] == cod) & (df_imig['nivel_jerarquia'] == nivel)
    return df_imig[m].set_index('fecha')['valor_millones_pesos'].sort_index().reindex(IDX)

# PIB nominal mensual (para % PIB): proxy EMAE x IPC calibrado trimestre a trimestre contra el PIB
# nominal de INDEC (que viene anualizado: /4 = PIB del trimestre). Trimestres sin PIB publicado
# usan el ultimo factor de calibracion.
pib_trim = (macro['pib_nominal_anualizado'] / 4).dropna()
pib_trim = pib_trim.groupby(pib_trim.index.to_period('Q')).first()
proxy    = (macro['emae_desest'].ffill(limit=FFILL_MAX) * ipc).dropna()
trim     = proxy.index.to_period('Q')
factor   = (pib_trim / proxy.groupby(trim).sum()).reindex(sorted(set(trim))).ffill()
pib_mensual = (proxy * factor.reindex(trim).values).reindex(IDX)
pib_12m     = pib_mensual.rolling(12).sum()

print(f'Series macro : {macro.shape[1]} | {macro.dropna(how="all").index.min():%Y-%m} a {macro.dropna(how="all").index.max():%Y-%m}')
print(f'IPC empalmado: {ipc.first_valid_index():%Y-%m} a {ipc.last_valid_index():%Y-%m}')
print(f'IMIG (repo)  : {df_imig.fecha.min():%Y-%m} a {df_imig.fecha.max():%Y-%m}')
print('PIB nominal anual (billones, INDEC):',
      (pib_mensual.groupby(pib_mensual.index.year).sum() / 1e6).loc[2019:2025].round(1).to_dict())

In [ ]:
# ── Celda 3: Construccion de las 16 variables ────────────────────────
def ia(s):
    # variacion interanual % (indice mensual continuo)
    return (s / s.shift(12) - 1) * 100

def suma12(s):
    return s.rolling(12).sum()

m = macro
recaud_real = (imig_serie('IVA_neto_reintegros', 2) + imig_serie('Debitos_creditos', 2)) / ipc
brecha = (m['ccl'] / m['a3500'] - 1) * 100
brecha[brecha.index < INICIO_CEPO] = 0.0
desocup = m['desocupacion'] * (100 if m['desocupacion'].max() < 1 else 1)  # la API la da en fraccion
tasa_real = ((1 + m['badlar'] / 1200) ** 12 / (1 + m['rem_12m'] / 100) - 1) * 100

# (codigo, nombre, pilar, signo [+1: mas alto = mejor], unidad, origen, serie)
VARS = [
    ('emae_ia',            'EMAE desest. var. i.a.',                  'Actividad',         +1, '%',       'INDEC (datos.gob.ar)', ia(m['emae_desest'])),
    ('recaudacion_ia',     'Recaudacion real IVA + Deb./Cred. (trim. movil) var. i.a.', 'Actividad', +1, '%', 'repo: IMIG + IPC', ia(recaud_real.rolling(3).sum())),
    ('sipa_ia',            'Asalariados privados registrados var. i.a.', 'Empleo e ingresos', +1, '%',     'SIPA (datos.gob.ar)', ia(m['sipa_asal_priv'])),
    ('salario_real_ia',    'Salario real registrado var. i.a.',       'Empleo e ingresos', +1, '%',       'INDEC + IPC repo',    ia(m['salario_registrado'] / ipc)),
    ('desocupacion',       'Tasa de desocupacion',                    'Empleo e ingresos', -1, '%',       'INDEC EPH (trimestral)', desocup),
    ('inflacion_3m',       'Inflacion 3 meses anualizada (log)',      'Precios',           -1, '% anual', 'repo: IPC',           400 * np.log(ipc / ipc.shift(3))),
    ('inflacion_esperada', 'Inflacion esperada 12 m REM (log)',       'Precios',           -1, '% anual', 'BCRA REM',            100 * np.log(1 + m['rem_12m'] / 100)),
    ('primario_pib',       'Resultado primario 12 m / PIB',           'Fiscal',            +1, '% PIB',   'repo: IMIG + PIB INDEC', suma12(imig_serie('RESULTADO_PRIMARIO', 0)) / pib_12m * 100),
    ('intereses_ingresos', 'Intereses netos / ingresos totales 12 m', 'Fiscal',            -1, '%',       'repo: IMIG',          suma12(imig_serie('Intereses_netos', 1)) / suma12(imig_serie('INGRESOS_TOTALES', 0)) * 100),
    ('reservas_meses_impo', 'Reservas brutas en meses de importaciones', 'Externo',         +1, 'meses',   'BCRA + INDEC ICA',    m['reservas_usd'] / (suma12(m['impo_usd']) / 12)),
    ('saldo_comercial',    'Saldo comercial 12 m',                    'Externo',           +1, 'MM USD',  'INDEC ICA (datos.gob.ar)', suma12(m['expo_usd'] - m['impo_usd']) / 1000),
    ('brecha',             'Brecha CCL / oficial A3500',              'Externo',           -1, '%',       'argentinadatos + BCRA', brecha),
    ('tcrm_desalineado',   'Desalineamiento del TCRM vs su mediana',  'Externo',           -1, '% abs.',  'BCRA ITCRM',          100 * (np.log(m['itcrm'] / m.loc[INICIO:, 'itcrm'].median())).abs()),
    ('riesgo_pais',        'Riesgo pais (EMBI)',                      'Financiero',        -1, 'pb',      'argentinadatos',      m['riesgo_pais']),
    ('credito_real_ia',    'Credito real al sector privado var. i.a.', 'Financiero',       +1, '%',       'BCRA + IPC repo',     ia(m['prestamos_privados'] / ipc)),
    ('tasa_real_desvio',   f'Distancia de la tasa real (BADLAR vs REM) a {TASA_REAL_NEUTRA:.0f}%', 'Financiero', -1, 'pp', 'BCRA', (tasa_real - TASA_REAL_NEUTRA).abs()),
]
meta = pd.DataFrame([v[:6] for v in VARS], columns=['codigo', 'nombre', 'pilar', 'signo', 'unidad', 'origen']).set_index('codigo')
X = pd.DataFrame({v[0]: v[6] for v in VARS}).reindex(IDX)

# Arrastre del ultimo dato (solo al final de cada serie, para series con rezago de publicacion)
OBS = X.notna()
for c in X:
    ult = X[c].last_valid_index()
    X.loc[ult:, c] = X.loc[ult:, c].ffill(limit=FFILL_MAX)

cobertura = OBS.loc[INICIO:].mean(axis=1)
ULTIMO_MES = cobertura[cobertura >= MIN_OBS_FIN].index.max()
X = X.loc[INICIO:ULTIMO_MES]
OBS = OBS.loc[INICIO:ULTIMO_MES]

print(f'Indice: {INICIO:%Y-%m} a {ULTIMO_MES:%Y-%m} (ultimo mes con >= {MIN_OBS_FIN:.0%} de variables con dato propio)')
print()
print(pd.DataFrame({'pilar': meta['pilar'],
                    'desde': X.apply(lambda s: s.first_valid_index()).dt.strftime('%Y-%m'),
                    'ult. dato propio': OBS.apply(lambda s: s[s].index.max()).dt.strftime('%Y-%m'),
                    f'valor {ULTIMO_MES:%Y-%m}': X.iloc[-1].round(1)}).to_string())

In [ ]:
# ── Celda 4: Normalizacion (z robusto) y agregacion por pilar ────────
# Escala = rango intercuartil / 1,349 (~ desvio estandar si fuera normal). Se prefiere al MAD
# porque la brecha tiene muchos ceros (sin cepo 2004-2011) y el MAD la dejaria casi en cero.
mediana = X.median()
escala  = (X.quantile(0.75) - X.quantile(0.25)) / 1.349
Z = ((X - mediana) / escala).clip(-CLIP_Z, CLIP_Z) * meta['signo']

P = Z.T.groupby(meta['pilar']).mean().T[PILARES]   # promedio de las variables disponibles del pilar
n_pil = P.notna().sum(axis=1)
indice = P.mean(axis=1).where(n_pil >= MIN_PILARES)
indice_sin_fiscal = P.drop(columns='Fiscal').mean(axis=1).where(P.drop(columns='Fiscal').notna().sum(axis=1) >= MIN_PILARES - 1)
contrib = P.div(n_pil, axis=0)                      # aporte de cada pilar: suma = indice

df_indice = pd.concat([indice.rename('indice'), indice_sin_fiscal.rename('indice_sin_fiscal'),
                       P, n_pil.rename('n_pilares')], axis=1)
df_indice.index.name = 'fecha'

ult = df_indice.iloc[-1]
hace12 = df_indice.loc[ULTIMO_MES - pd.DateOffset(months=12)]
print(f'INDICE MACRO {ULTIMO_MES:%b-%Y}: {ult.indice:+.2f}  (hace 12 meses: {hace12.indice:+.2f})')
print(f'Sin fiscal               : {ult.indice_sin_fiscal:+.2f}  (hace 12 meses: {hace12.indice_sin_fiscal:+.2f})')
print()
print(pd.DataFrame({'actual': ult[PILARES], 'hace 12 m': hace12[PILARES]}).round(2).to_string())

In [ ]:
# ── Celda 5: Grafico 01 - Indice historico ───────────────────────────
fig, ax = plt.subplots(figsize=(18, 6.5))
s = df_indice['indice_sin_fiscal']
ax.fill_between(s.index, 0, s, where=s >= 0, color='#27ae60', alpha=0.25, lw=0)
ax.fill_between(s.index, 0, s, where=s < 0,  color='#c0392b', alpha=0.25, lw=0)
ax.plot(s.index, s, color='#2c3e50', lw=1.4, label='Indice sin fiscal (comparable 2004-hoy)')
ax.plot(df_indice.index, df_indice['indice'], color='#8e44ad', lw=1.8, label='Indice completo (con pilar fiscal, desde 2019)')
ax.axhline(0, color='black', lw=0.8)
sombrear_gobiernos(ax, INICIO, ULTIMO_MES + pd.DateOffset(months=1))
ax.set_xlim(INICIO, ULTIMO_MES + pd.DateOffset(months=1))
ax.set_ylabel('Desvios respecto de lo tipico 2004-hoy (z)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='lower left', fontsize=9, frameon=False)
plt.tight_layout()
save_fig(fig, '01_indice_macro_historico')
plt.show()

In [ ]:
# ── Celda 6: Grafico 02 - Aporte de cada pilar (ultimos 36 meses) ────
c = contrib.iloc[-36:]
fig, ax = plt.subplots(figsize=(18, 6.5))
pos_b = np.zeros(len(c)); neg_b = np.zeros(len(c))
x = np.arange(len(c))
for p in PILARES:
    v = c[p].fillna(0).values
    ax.bar(x, np.where(v > 0, v, 0), bottom=pos_b, color=COLOR_PILAR[p], alpha=0.85, width=0.8, label=p)
    ax.bar(x, np.where(v < 0, v, 0), bottom=neg_b, color=COLOR_PILAR[p], alpha=0.85, width=0.8)
    pos_b += np.where(v > 0, v, 0); neg_b += np.where(v < 0, v, 0)
ax.plot(x, df_indice['indice'].iloc[-36:].values, 'o-', color='black', lw=1.8, ms=4, label='Indice')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x[::3]); ax.set_xticklabels([d.strftime('%b-%y') for d in c.index[::3]], rotation=45, ha='right')
ax.set_ylabel('Aporte al indice (z)')
ax.grid(axis='y', alpha=0.25)
ax.legend(ncol=7, fontsize=8, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.14))
plt.tight_layout()
save_fig(fig, '02_aporte_pilares')
plt.show()

In [ ]:
# ── Celda 7: Grafico 03 - Mapa de calor de variables (ultimos 24 meses) ─
orden = meta.sort_values('pilar', key=lambda s: s.map(PILARES.index)).index
H = Z[orden].iloc[-24:].T
fig, ax = plt.subplots(figsize=(18, 7.5))
im = ax.imshow(H.values, aspect='auto', cmap='RdYlGn', vmin=-CLIP_Z, vmax=CLIP_Z)
ax.set_yticks(range(len(H))); ax.set_yticklabels([f"{meta.loc[k, 'pilar']} | {meta.loc[k, 'nombre']}" for k in H.index], fontsize=8)
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels([d.strftime('%b-%y') for d in H.columns], rotation=45, ha='right', fontsize=8)
arr = (~OBS[orden].iloc[-24:].T).values
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.1f}' + ('*' if arr[i, j] else ''), ha='center', va='center', fontsize=6.5,
                    color='white' if abs(v) > 2 else 'black')
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='z (verde = mejor)')
ax.set_xlabel('* = ultimo dato arrastrado (serie todavia no publicada para ese mes)', fontsize=8)
plt.tight_layout()
save_fig(fig, '03_mapa_calor_variables')
plt.show()

In [ ]:
# ── Celda 8: Grafico 04 - Validacion contra el indice de miseria ─────
miseria = ((ipc / ipc.shift(12) - 1) * 100 + desocup.ffill(limit=FFILL_MAX)).loc[INICIO:ULTIMO_MES]
comp = pd.concat([df_indice['indice_sin_fiscal'], miseria.rename('miseria')], axis=1).dropna()
corr = comp.corr(method='spearman').iloc[0, 1]

fig, ax = plt.subplots(figsize=(18, 5.5))
ax.plot(comp.index, comp['indice_sin_fiscal'], color='#2c3e50', lw=1.5, label='Indice macro sin fiscal (izq.)')
ax.axhline(0, color='black', lw=0.6)
ax.set_ylabel('z')
ax2 = ax.twinx()
ax2.plot(comp.index, comp['miseria'], color='#c0392b', lw=1.2, alpha=0.8, label='Indice de miseria: inflacion i.a. + desocupacion (der., invertido)')
ax2.set_yscale('log'); ax2.invert_yaxis()
ax2.set_yticks([10, 20, 30, 50, 100, 200, 300])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}'))
ax2.yaxis.set_minor_formatter(plt.NullFormatter())
ax2.set_ylabel('% (escala invertida)')
sombrear_gobiernos(ax, comp.index.min(), comp.index.max())
ax.legend(handles=ax.get_lines()[:1] + ax2.get_lines(), loc='lower left', fontsize=8, frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_xlim(comp.index.min(), comp.index.max())
plt.tight_layout()
save_fig(fig, '04_validacion_miseria')
plt.show()
print(f'Correlacion de Spearman indice sin fiscal vs miseria: {corr:+.2f} (se espera negativa)')

# Promedio del indice por gestion
lims = [pd.Timestamp(g[0]) for g in GOBIERNOS] + [ULTIMO_MES + pd.DateOffset(days=1)]
por_gob = []
for i, (ini, nombre) in enumerate(GOBIERNOS):
    t = df_indice[(df_indice.index >= max(pd.Timestamp(ini), INICIO)) & (df_indice.index < lims[i + 1])]
    por_gob.append({'gestion': nombre, 'desde': t.index.min().strftime('%Y-%m'), 'hasta': t.index.max().strftime('%Y-%m'),
                    'indice_sin_fiscal': t['indice_sin_fiscal'].mean(), 'indice': t['indice'].mean(),
                    **{p: t[p].mean() for p in PILARES}})
df_gob = pd.DataFrame(por_gob).set_index('gestion')
print('\nPromedio por gestion (z):')
print(df_gob.round(2).to_string())

In [ ]:
# ── Celda 9: Exportar Excel + ZIP ────────────────────────────────────
EXCEL_FILE = 'indice_macro.xlsx'
ZIP_FILE   = 'indice_macro.zip'

leeme = pd.DataFrame({'Contenido': [
    f'Indice Macroeconomico Argentina (IMA), mensual {INICIO:%Y-%m} a {ULTIMO_MES:%Y-%m}.',
    '0 = situacion tipica 2004-hoy; positivo = mejor; negativo = peor (desvios robustos, z).',
    'Indice = promedio de 6 pilares; cada pilar = promedio de sus variables (z robusto: (x - mediana) / (IQR/1,349), recortado a +/-3).',
    'indice_sin_fiscal: sin el pilar fiscal (que arranca en dic-2019); comparable en toda la serie.',
    'Indice: indice, indice sin fiscal, pilares y cantidad de pilares con dato.',
    'Z_variables: z-score de cada variable (signo: mas alto = mejor).',
    'Variables: valores de las 16 variables transformadas (ver Metodologia para unidades).',
    'Metodologia: definicion, pilar, signo, unidad, origen, mediana y escala (IQR/1,349) de cada variable.',
    'Por_gobierno: promedio del indice y de los pilares por gestion.',
    'Datos_crudos: series externas sin transformar (data/reference/macro_mensual.csv).',
    f'Series con rezago: el ultimo dato se arrastra hasta {FFILL_MAX} meses (ver * en el grafico 03).',
    'Inflacion 2007-2015: periodo de intervencion del INDEC; el IPC empalmado usa la serie del BCRA.',
]})
metodo = meta.assign(mediana=mediana, escala=escala).reset_index()

with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as xw:
    leeme.to_excel(xw, sheet_name='Leeme', index=False)
    df_indice.round(3).to_excel(xw, sheet_name='Indice')
    Z.round(3).rename_axis('fecha').to_excel(xw, sheet_name='Z_variables')
    X.round(3).rename_axis('fecha').to_excel(xw, sheet_name='Variables')
    metodo.round(3).to_excel(xw, sheet_name='Metodologia', index=False)
    df_gob.round(3).to_excel(xw, sheet_name='Por_gobierno')
    macro.loc[INICIO:].rename_axis('fecha').to_excel(xw, sheet_name='Datos_crudos')

with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(EXCEL_FILE)
    for g in GRAFICOS:
        z.write(g)
print(f'ZIP: {ZIP_FILE} ({len(GRAFICOS)} graficos + {EXCEL_FILE})')

if IN_COLAB:
    from google.colab import files
    files.download(ZIP_FILE)